In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X, y = load_breast_cancer(return_X_y=True)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

baseline = LogisticRegression(max_iter=10000)
baseline.fit(X_train, y_train)
print("Baseline accuracy:", round(baseline.score(X_val, y_val), 4))

scores = cross_val_score(baseline, X_train, y_train, cv=5)
print("CV mean accuracy:", round(scores.mean(), 4))

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000)),
])
pipe.fit(X_train, y_train)
print("Pipeline accuracy:", round(pipe.score(X_val, y_val), 4))

params = [
    {
        "model": [LogisticRegression(max_iter=10000, solver="liblinear")],
        "model__C": [0.01, 0.1, 1, 3, 10, 30, 100],
        "model__penalty": ["l1", "l2"],
    },
    {
        "model": [SVC()],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.001, 0.01, 0.1],
    },
]
grid = GridSearchCV(pipe, params, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best model:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))
print("Tuned validation accuracy:", round(grid.score(X_val, y_val), 4))

print("Test accuracy:", round(grid.score(X_test, y_test), 4))

Baseline accuracy: 0.9474
CV mean accuracy: 0.9618
Pipeline accuracy: 0.9912
Best model: {'model': LogisticRegression(max_iter=10000, solver='liblinear'), 'model__C': 0.1, 'model__penalty': 'l2'}
Best CV accuracy: 0.9824
Tuned validation accuracy: 0.9737
Test accuracy: 0.9825


C:\Users\LAKSHAN\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
